# betterproto_fw smoke test

In [ ]:
# Example 1: define a simple Message subclass and verify field defaults + repr.
from dataclasses import dataclass

import betterproto


@dataclass
class Point(betterproto.Message):
    x: float = betterproto.double_field(1)
    y: float = betterproto.double_field(2)
    label: str = betterproto.string_field(3)


p = Point(x=1.5, y=-2.25, label="origin")
print(p)
assert p.x == 1.5
assert p.y == -2.25
assert p.label == "origin"

default = Point()
assert default.x == 0.0
assert default.y == 0.0
assert default.label == ""

In [ ]:
# Example 2: serialize a Message to wire bytes and deserialize back, asserting round-trip equality.
from dataclasses import dataclass
from typing import List

import betterproto


@dataclass
class Person(betterproto.Message):
    name: str = betterproto.string_field(1)
    age: int = betterproto.int32_field(2)
    emails: List[str] = betterproto.string_field(3)


original = Person(name="Ada", age=36, emails=["ada@example.com", "ada.l@example.org"])
wire = bytes(original)
print(f"serialized {len(wire)} bytes: {wire!r}")

decoded = Person().parse(wire)
assert decoded.name == "Ada"
assert decoded.age == 36
assert decoded.emails == ["ada@example.com", "ada.l@example.org"]
assert decoded == original

In [ ]:
# Example 3: JSON serialization (to_dict / from_dict) using the camelCase casing convention.
from dataclasses import dataclass

import betterproto


@dataclass
class UserProfile(betterproto.Message):
    user_id: int = betterproto.int64_field(1)
    display_name: str = betterproto.string_field(2)
    is_admin: bool = betterproto.bool_field(3)


u = UserProfile(user_id=42, display_name="alice", is_admin=True)
as_dict = u.to_dict()
print(as_dict)
assert as_dict == {"userId": "42", "displayName": "alice", "isAdmin": True}

restored = UserProfile().from_dict(as_dict)
assert restored == u

json_str = u.to_json()
assert "alice" in json_str
print(json_str)

In [ ]:
# Example 4: enum fields + Casing helpers (snake_case <-> camelCase).
from dataclasses import dataclass

import betterproto
from betterproto import Casing


class Color(betterproto.Enum):
    UNKNOWN = 0
    RED = 1
    GREEN = 2
    BLUE = 3


@dataclass
class Pixel(betterproto.Message):
    color: "Color" = betterproto.enum_field(1)
    alpha: float = betterproto.float_field(2)


px = Pixel(color=Color.GREEN, alpha=0.5)
wire = bytes(px)
decoded = Pixel().parse(wire)
assert decoded.color == Color.GREEN
assert abs(decoded.alpha - 0.5) < 1e-6

snake = px.to_dict(casing=Casing.SNAKE)
camel = px.to_dict(casing=Casing.CAMEL)
print(f"snake: {snake}")
print(f"camel: {camel}")
assert "color" in snake
assert decoded == px